# 01 - Data Exploration

This notebook loads Oxford-IIIT Pet from TensorFlow Datasets, checks the metadata, visualizes images and trimap masks, and saves report-ready figures.

## Google Colab Git Setup

Run the next cell only when using Google Colab. Set `REPO_URL` to your GitHub repository URL, then the cell clones or pulls the repository into `/content/PetVision-DeepLearning` and installs dependencies. If you run locally, skip it.


In [ ]:
# Colab-only Git setup. Skip this cell when running locally.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/andri-10/computer_vision.git"

REPO_DIR = Path("/content/computer_vision")
PROJECT_DIR = REPO_DIR / "PetVision-DeepLearning"

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    if REPO_DIR.exists():
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "origin", "main"])
    else:
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

    if not PROJECT_DIR.exists():
        raise FileNotFoundError(f"Project folder not found: {PROJECT_DIR}")

    os.chdir(PROJECT_DIR)

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"
    ])

    print("Colab repo root:", REPO_DIR)
    print("Colab project root:", os.getcwd())
else:
    print("Not running in Google Colab. Continue with the local setup cells below.")

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

for path in [
    "models",
    "results/classification",
    "results/segmentation",
    "results/gradcam",
    "results/figures",
]:
    (PROJECT_ROOT / path).mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

from src.data_loader import load_oxford_pet, get_label_names, save_label_mapping
from src.visualization import colorize_mask

In [ ]:
datasets, info = load_oxford_pet(with_info=True)
label_names = get_label_names(info)
label_mapping = save_label_mapping(label_names, PROJECT_ROOT / "results/figures/label_mapping.json")

print(info)
print(f"Number of classes: {len(label_names)}")
print(f"Train examples: {info.splits['train'].num_examples}")
print(f"Test examples: {info.splits['test'].num_examples}")
print(label_names[:10])

In [ ]:
# Display random pet images with breed names.
samples = list(datasets["train"].shuffle(1000, seed=42).take(12).as_numpy_iterator())
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, sample in zip(axes.ravel(), samples):
    ax.imshow(sample["image"])
    ax.set_title(label_names[int(sample["label"])], fontsize=9)
    ax.axis("off")
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "results/figures/sample_images.png", dpi=160)
plt.show()

In [ ]:
# Show image and segmentation mask together.
mask_samples = list(datasets["train"].shuffle(1000, seed=7).take(4).as_numpy_iterator())
fig, axes = plt.subplots(4, 3, figsize=(10, 14))
for row, sample in enumerate(mask_samples):
    image = sample["image"]
    raw_mask = sample["segmentation_mask"]
    converted_mask = np.clip(np.squeeze(raw_mask).astype(np.int32) - 1, 0, 2)
    axes[row, 0].imshow(image)
    axes[row, 0].set_title(label_names[int(sample["label"])])
    axes[row, 1].imshow(np.squeeze(raw_mask), cmap="viridis")
    axes[row, 1].set_title("Raw trimap labels 1/2/3")
    axes[row, 2].imshow(colorize_mask(converted_mask))
    axes[row, 2].set_title("Converted mask 0/1/2")
    for col in range(3):
        axes[row, col].axis("off")
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "results/figures/sample_masks.png", dpi=160)
plt.show()

In [ ]:
# Class distribution for train and test splits.
def labels_from_split(ds):
    return [int(item["label"].numpy()) for item in tfds.as_numpy(ds.map(lambda x: {"label": x["label"]}))]

train_labels = [int(x["label"]) for x in datasets["train"].as_numpy_iterator()]
test_labels = [int(x["label"]) for x in datasets["test"].as_numpy_iterator()]
counts = pd.DataFrame({
    "class_name": label_names,
    "train": np.bincount(train_labels, minlength=len(label_names)),
    "test": np.bincount(test_labels, minlength=len(label_names)),
})
counts.to_csv(PROJECT_ROOT / "results/figures/class_distribution.csv", index=False)

fig, ax = plt.subplots(figsize=(15, 6))
ax.bar(np.arange(len(label_names)) - 0.2, counts["train"], width=0.4, label="train")
ax.bar(np.arange(len(label_names)) + 0.2, counts["test"], width=0.4, label="test")
ax.set_xticks(np.arange(len(label_names)))
ax.set_xticklabels(label_names, rotation=90, fontsize=8)
ax.set_ylabel("Images")
ax.set_title("Oxford-IIIT Pet class distribution")
ax.legend()
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "results/figures/class_distribution.png", dpi=160)
plt.show()

In [ ]:
# Inspect original image sizes.
sizes = []
for sample in datasets["train"].take(500).as_numpy_iterator():
    height, width = sample["image"].shape[:2]
    sizes.append((height, width))
sizes_df = pd.DataFrame(sizes, columns=["height", "width"])
print(sizes_df.describe())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(sizes_df["height"], bins=30)
axes[0].set_title("Image heights")
axes[1].hist(sizes_df["width"], bins=30)
axes[1].set_title("Image widths")
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "results/figures/image_size_distribution.png", dpi=160)
plt.show()

## Notes for Report

- The dataset has 37 classes and official train/test splits.
- Images have variable sizes, so all training pipelines resize images.
- Segmentation masks are trimaps with labels 1, 2, and 3, converted later to 0, 1, and 2.